In [0]:
%sql
USE CATALOG ecommerce;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
from pyspark.sql import functions as F

# Configuração dos schemas utilizados
catalogo      = "ecommerce"
silver_schema = "silver"
gold_schema   = "gold"

In [0]:
from pyspark.sql import functions as F

df_pedidos  = spark.table(f"{catalogo}.{silver_schema}.fat_pedidos")
df_itens    = spark.table(f"{catalogo}.{silver_schema}.fat_itens_pedidos")
df_produtos = spark.table(f"{catalogo}.{silver_schema}.dim_produtos")
df_cotacao  = spark.table(f"{catalogo}.{silver_schema}.dim_cotacao_dolar")

# Join para enriquecer itens com data do pedido, categoria e cotação
df_base = (df_itens
    .join(df_pedidos,  on="id_pedido",  how="left")
    .join(df_produtos, on="id_produto", how="left")
    .withColumn("data_cotacao", F.to_date("data_pedido"))
    .join(df_cotacao, on="data_cotacao", how="left")
    .withColumn("ano_venda", F.year("data_pedido"))
    .withColumn("mes_venda", F.month("data_pedido"))
)

# Agrega por ano, mês e categoria conforme granularidade exigida
df_vendas = (df_base
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        # Contagem distinta evita dupla contagem de pedidos com múltiplos itens
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("id_item").alias("qtd_itens_vendidos"),
        # Soma dos preços dos itens — granularidade correta para receita
        F.round(F.sum("preco_BRL"), 2).alias("receita_total_brl"),
        # Converte cada item para USD via cotação do dia do pedido
        F.round(F.sum(F.col("preco_BRL") / F.col("cotacao_dolar")), 2).alias("receita_total_usd"),
        # Ticket médio = receita total / pedidos distintos
        F.round(F.sum("preco_BRL") / F.countDistinct("id_pedido"), 2).alias("ticket_medio_brl")
    )
)

# Apenas para validação
#df_vendas.display()

(df_vendas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{gold_schema}.fat_vendas_comercial"))

print(f"✓ fat_vendas_comercial ({df_vendas.count()} linhas)")

ano_venda,mes_venda,categoria_produto,total_pedidos,qtd_itens_vendidos,receita_total_brl,receita_total_usd,ticket_medio_brl
2017,6,informatica_acessorios,219,261,37007.08,11250.77,168.98
2018,3,cama_mesa_banho,670,798,69256.39,21105.32,103.37
2017,3,cama_mesa_banho,249,289,25773.02,8239.87,103.51
2017,8,informatica_acessorios,297,350,35025.72,11118.94,117.93
2017,7,eletronicos,87,97,7346.84,2291.98,84.45
2017,12,informatica_acessorios,252,287,37880.65,11508.15,150.32
2017,5,eletronicos,62,68,6709.11,2100.24,108.21
2018,7,alimentos,59,64,3378.17,887.31,57.26
2018,5,cine_foto,17,18,2277.29,630.54,133.96
2017,1,automotivo,30,34,5218.53,1642.8,173.95


✓ fat_vendas_comercial (1283 linhas)


In [0]:
df_top_prod = (
    df_base
    .groupBy("nome_produto", "categoria_produto")
    .agg(F.count("*").alias("qtd_vendida"))
    .orderBy(F.desc("qtd_vendida"))
    .limit(5)
)

df_top_prod.display()

nome_produto,categoria_produto,qtd_vendida
Secador de Cabelo,beleza_saude,1076
Toalha de Banho,cama_mesa_banho,919
Protetor Solar,beleza_saude,917
Travesseiro,cama_mesa_banho,839
Colar,relogios_presentes,804


In [0]:
df_bottom_prod = (
    df_base
    .groupBy("nome_produto", "categoria_produto")
    .agg(F.count("*").alias("qtd_vendida"))
    .orderBy(F.asc("qtd_vendida"))
    .limit(5)
)

df_bottom_prod.display()

nome_produto,categoria_produto,qtd_vendida
Produto Genérico Preto,moveis_quarto,1
Item Básico Premium,fashion_calcados,1
Peça de Reposição Preto,telefonia_fixa,1
Item Básico Verde,industria_comercio_e_negocios,1
Peça de Reposição Plus,fashion_calcados,1


In [0]:
from pyspark.sql import functions as F

df_reviews = spark.table(f"{catalogo}.{silver_schema}.fat_avaliacoes_pedidos")
df_itens   = spark.table(f"{catalogo}.{silver_schema}.fat_itens_pedidos")
df_prod    = spark.table(f"{catalogo}.{silver_schema}.dim_produtos")
df_vend    = spark.table(f"{catalogo}.{silver_schema}.dim_vendedores")

# garante 1 avaliação por pedido
df_reviews = df_reviews.dropDuplicates(["id_pedido"])

# Base com joins
df_base = (
    df_reviews
    .join(df_itens, "id_pedido")
    .join(df_prod, "id_produto")
    .join(df_vend, "id_vendedor")
)

In [0]:
df_avaliacoes = (
    df_base
    .groupBy("categoria_produto", "nome_vendedor", "estado")
    .agg(
        F.count("*").alias("total_avaliacoes"),
        
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        
        F.sum(
            F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)
        ).alias("total_avaliacoes_positivas"),
        
        F.sum(
            F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)
        ).alias("total_avaliacoes_negativas")
    )
    .withColumn(
        "percentual_satisfacao",
        F.round(
            (F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100,
            2
        )
    )
)

(df_avaliacoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{gold_schema}.fat_avaliacoes_clientes")
)

In [0]:
# TABELA PARA RANKING DE PRODUTOS
df_rank_prod = (
    df_base
    .groupBy("nome_produto")
    .agg(
        F.avg("nota_avaliacao").alias("media"),
        F.count("*").alias("qtd")
    )
)

In [0]:
# PRODUTO MAIS BEM AVALIADO
df_rank_prod.orderBy(
    F.desc("media"),  # maior nota primeiro
    F.desc("qtd")     # desempate por quantidade
).limit(1).display()

nome_produto,media,qtd
Caneta Esferográfica Avançado,5.0,18


In [0]:
# PRODUTO MENOS BEM AVALIADO
df_rank_prod.orderBy(
    F.asc("media"),   # menor nota primeiro
    F.desc("qtd")
).limit(1).display()

nome_produto,media,qtd
Conjunto de Pincéis Ultra,1.0,7


In [0]:
# TABELA PARA RANKING DE VENDEDORES
df_rank_vend = (
    df_base
    .groupBy("nome_vendedor")
    .agg(
        F.avg("nota_avaliacao").alias("media"),
        F.count("*").alias("qtd")
    )
)

In [0]:
# VENDEDOR MAIS BEM AVALIADO
df_rank_vend.orderBy(
    F.desc("media"),
    F.desc("qtd")
).limit(1).display()

nome_vendedor,media,qtd
Luiz Otávio Abreu,5.0,34


In [0]:
# VENDEDOR MENOS BEM AVALIADO
df_rank_vend.orderBy(
    F.asc("media"),
    F.desc("qtd")
).limit(1).display()

nome_vendedor,media,qtd
Sra. Fernanda Santos,1.0,8
